# Make_Sycamore_Model

**Lower Sycamore Wash (USGS gauge 09510200), water year 2017: builds the tRIBS model inputs and writes the tRIBS input file.**

This notebook uses [pytRIBS](https://pypi.org/project/pytRIBS/) (Raming et al., 2025) to assemble a [tRIBS](https://tribshms.readthedocs.io/en/latest/) model. It started as the South Mountain Fan (SMF) notebook, which was built from the pytRIBS sandbox example, and was rewritten for Sycamore on 2026-10-15. On 2026-10-22 it was reorganized: every import moved into one cell, headings and explanations were added, and the cells that compare finished runs with the gauge moved to their own notebook, `Compare_Sycamore_Runs.ipynb`. The older comparison cells, which read the unrouted `Srf` column, were removed.

### What it does
1. Imports and Settings.
2. Checks the inputs you prepared by hand, and sets up the model folders (Project).
3. Mesh: delineates the basin from the DEM and builds the TIN mesh.
4. Soil: downloads SOLUS, turns it into tRIBS soil grids, applies the Ks, PsiB and f multipliers, makes the starting water table, and copies everything into the model folder.
5. Met: points tRIBS at the NLDAS weather stations and the hourly radar rain.
6. Land: copies in the land cover map and table.
7. Model: sets the options, chooses output nodes, checks every file path, writes the input file, and reads it back.

**Run tRIBS** afterwards from the terminal (the command is in the "Done" cell), then open `Compare_Sycamore_Runs.ipynb` to compare the run with the gauge.

### Folder rules
- Run this notebook from `workspaces/Sycamore/lab/`.
- Anything you prepare by hand lives in `../init_data/`. This notebook creates and fills `data/model/` itself, so **never place files in `data/model/` by hand**.
- `data/`, `results/` and `restart/` inside `lab/` are generated and are never committed.

### Inputs that must exist before you start
| Input | Where | Made by |
|---|---|---|
| DEM (10 m, EPSG 26912, 5 km buffer around the basin) | `../init_data/Sycamore_DEM.tif` | ArcGIS Pro |
| Land cover map | `../init_data/LandUse.asc` | `make_sycamore_landuse_map.py` |
| Land cover table | `../init_data/land_use_params.ldt` | `make_sycamore_landuse_table.py` |
| Weather stations (3 NLDAS points) | `../init_data/met/Master_Met.sdf`, plus the `.mdf` files it lists | `Generate_Met_Forcing_Sycamore.ipynb` |
| Radar rainfall grids (hourly, end-of-hour labels) | a folder outside the repo, found automatically (see Settings) | `build_mrms_radar_rainfall_tribs6.py` |

The SOLUS soil grids are downloaded by this notebook, so there is nothing to prepare for them.

### How to run it
- **First time, or after a kernel restart:** run everything from the top. The delineation re-runs and gives the same answer.
- **A new calibration run:** edit the **Settings** cell (change the value you are testing, and give the run new `INPUT_FILE` and `RESULTS_NAME` values), then run from the Settings cell to the end ("Execute Cell and Below" from the Settings cell works). The mesh is reused when its files exist, and the SOLUS download and the soil workflow are skipped when their files exist, so a rebuild is faster than the first build.
- **A different test window** (one week, a monsoon month, the full year): change `RUN_CHOICE` in Settings, then rerun from the Settings cell down.
- The SOLUS download and the soil workflow take several minutes the first time.

## Imports

An *import* makes a Python library available to the notebook. **All imports for this notebook are in the next cell**, so run it first in every session (after a kernel restart too). It also loads the pytRIBS classes (`Project`, `Mesh`, `Soil`, `Met`, `Land`, `Model`, `InOut`) and prints the installed pytRIBS version.

In [ ]:
# Standard library
import os
import csv
import glob
import re
import shutil
import time
from datetime import datetime, timedelta
from importlib import metadata

# Numbers, tables and maps
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
from shapely.ops import unary_union

# Figures and tables in the notebook
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

# pytRIBS. This also brings in Project, Mesh, Soil, Met, Land, Model, InOut and Aux.
from pytRIBS.classes import *

try:
    print('pytRIBS version:', metadata.version('pytRIBS'))
except metadata.PackageNotFoundError:
    print('pytRIBS version: not found by importlib.metadata')

## Settings

Every number that is specific to this basin or this run is in the next cell. The rest of the notebook only reads these values, so this is the cell to edit before a new run.

Things to know before changing anything:
- **Run window.** Pick one entry of `RUNS` with `RUN_CHOICE`. The radar files are labeled by the **end** of each hour, so a run that starts at 10/01/2016 00:00 first asks for `radar_1001201601.asc`. The weather files cover 10/01/2016 00:00 to 09/30/2017 23:00.
- **Calibration levers.** `KS_MULT`, `PSIB_MULT` and `F_MULT` multiply the SOLUS grids for saturated conductivity (Ks), air-entry pressure (PsiB) and conductivity decay with depth (f); 1 means unchanged. `KINEMVELCOEF`, `FLOWEXP`, `CHANNELROUGHNESS` and `CHANNELWIDTHCOEFF` are the routing values. `OPTPERCOLATION` turns channel loss on (1) or off (0); it is off in the working runs. The routing and channel values started as SMF's and some have since been changed by the Sycamore calibration (for example `KINEMVELCOEF`), so treat all of them as calibration choices, not as measured Sycamore values. The channel width coefficient in particular needs a recheck for a basin this large (question 6 for Josh).
- **Naming a run.** Give every run new `INPUT_FILE` and `RESULTS_NAME` values (each name appears once in the cell). The pattern used so far is: window, Ks multiplier, cv, other changes, then the loss setting last, for example `Sycamore_octmar_ks0p2_cv6_psib0p5_nocl` (the `p` stands for a decimal point; `nocl` = no channel loss).
- **Snow is on** (`OPTSNOW = 1`), following Josh's advice of 2026-10-15.

In [ ]:
# =============================================================================================
# SETTINGS: everything specific to this basin or this run
# =============================================================================================

# ---- Project ---------------------------------------------------------------------------------
NAME = 'Sycamore'
EPSG = 26912                              # NAD83 / UTM zone 12N

# ---- Watershed delineation (Step 3) ----------------------------------------------------------
DEM = '../init_data/Sycamore_DEM.tif'
OUTLET_X, OUTLET_Y = 449721, 3728390      # USGS 09510200 in UTM 12N (m)
SNAP_DIS = 100                            # m: how far the outlet may move to land on the stream line
THRESHOLD_AREA = 2e4                      # a COUNT OF DEM CELLS, not m2: cells that must drain to a point
                                          # before it counts as stream. 2e4 cells x 100 m2 = 2 km2 here.
VERBOSE_WBT = False                       # True prints the WhiteboxTools messages

# ---- Mesh (Step 4) ---------------------------------------------------------------------------
MESH_THRESHOLD = 0.9                      # 0 to 1. Lower keeps more terrain points (denser mesh).
BOUNDARY_BUFFER_DIST = 50.0               # m: watershed buffered outward to make the no-flow boundary
BOUNDARY_SPACING = 100.0                  # m: node spacing on the boundary ring
STREAM_POINT_SPACING = 100.0              # m: node spacing along streams
STREAM_CLEAR_RADIUS = 2.0                 # m: interior points this close to a stream are removed
QUALITY_OPTS = 'q15a50000'                # Triangle options. q15 = no interior angle under 15 degrees.
                                          # The "a" part does not limit triangle size here (pytRIBS rescales
                                          # coordinates first; from the source, not tested by rerunning).
MAX_CANDIDATE_POINTS = 15000              # safety stop. On the 7.8 GiB Codespace 12,216 fit and 18,850 crashed.

# ---- Soils (Step 5) --------------------------------------------------------------------------
SOLUS_DIR = '../init_data'                # SOLUS grids are written to {SOLUS_DIR}/solus
SOLUS_DEPTHS = ['0-5cm', '5-15cm', '15-30cm']
SOLUS_VARIABLES = ['sandtotal', 'silttotal', 'claytotal', 'dbovendry']
GW_FRACTION = 0.95                        # starting water table = this fraction of the depth to bedrock
KS_MULT = 0.2                             # whole-basin multiplier on surface Ks (1 = SOLUS values unchanged)
PSIB_MULT = 0.5                           # whole-basin multiplier on air-entry pressure (1 = SOLUS values unchanged)
F_MULT = 0.5                              # whole-basin factor on f; 1 = unchanged
BEDROCK_ADD_M = 1.0                       # meters added to every SOLUS depth-to-bedrock cell (0 = no change)
SOIL_CONSTANTS = {'As': 1, 'Au': 1, 'ks': 0.7, 'Cs': 1.4e6}   # table columns no soil grid supplies
REBUILD_SOIL = False                      # True re-runs the slow soil workflow even if its files exist

# ---- Land cover (Step 6) ---------------------------------------------------------------------
LAND_MAP = '../init_data/LandUse.asc'
LAND_TABLE = '../init_data/land_use_params.ldt'
LAND_CLASS_NAMES = {1: 'Woodland', 2: 'Desert scrub', 3: 'Chaparral and oak shrub', 4: 'Bare rock',
                    5: 'Recently burned or logged', 6: 'Developed', 7: 'Riparian and water', 8: 'Grassland'}

# ---- Weather and rain (Step 7) ---------------------------------------------------------------
MET_STATIONS = '../init_data/met/Master_Met.sdf'
RADAR_ROOT = '~/mrms_wk_wy2017'           # searched for the folder holding the END-labeled radar files
RADAR_DIR = None                          # None = find it under RADAR_ROOT. Or give the folder explicitly.
RADAR_BASENAME = 'radar_'
RADAR_EXT = 'asc'
RADAR_FIRST_FILE = 'radar_1001201601.asc' # first end-labeled hour: 10/01/2016 00:00 to 01:00
RADAR_LAST_FILE = 'radar_1001201700.asc'  # only the end-labeled set has this one (hour ending 10/01/2017 00:00)
RAIN_INTERVAL_HR = 1                      # tRIBS 6.0.0 gridded rain is hourly only
UTC_OFFSET = -7                           # Arizona: MST all year, no daylight saving

# ---- Run window ------------------------------------------------------------------------------
RUNS = {
    'week':          ('10/01/2016/00/00', 168),    # first test: does tRIBS read the mesh and files, how fast is it
    'jan_storm_week': ('01/18/2017/00/00', 168),   # includes the Jan 21 2017 peak (7,800 cfs). Starts from the
                                                   # 95%-of-bedrock water table, so judge behavior, not accuracy.
    'monsoon_month': ('07/01/2017/00/00', 744),    # July 2017, the wettest month
    'full_year':     ('10/01/2016/00/00', 8760),   # all of WY2017 (weather ends 1 h before this run does; see Met section)
    'oct_to_jan':    ('10/01/2016/00/00', 2784),   # spin-up: Oct 1 to Jan 25 (116 days); the Jan storms come at the end
    'oct_to_mar':    ('10/01/2016/00/00', 4368),   # Oct 1 to Apr 1 (182 days): dry months, Jan flood, Feb and Mar storms
}
RUN_CHOICE = 'oct_to_mar'
START_DATE, RUN_HOURS = RUNS[RUN_CHOICE]       # format MM/DD/YYYY/HH/MM and length in hours

# ---- Model options and calibration starting values (SMF's values) ---------------------------
OPTSNOW = 1                               # 1 = snow module on
OPTPERCOLATION = 0                        # 1 = constant channel transmission loss, 0 = none
CHANNELCONDUCTIVITY = 5                  # mm/hr, streambed (called cc in the SMF work)
KINEMVELCOEF = 6                          # hillslope velocity multiplier
FLOWEXP = 0.3                             # how velocity scales with discharge (usual range 0.3 to 0.6)
CHANNELROUGHNESS = 0.04                   # Manning's n for channels
CHANNELWIDTHCOEFF = 2.33                  # channel width coefficient. RECHECK for ~425 km2.

# ---- Outputs ---------------------------------------------------------------------------------
INPUT_FILE = 'Sycamore_octmar_ks0p2_cv6_psib0p5_f0p5_nocl.in'               # written in lab/
RESULTS_NAME = 'Sycamore_octmar_ks0p2_cv6_psib0p5_f0p5_nocl'                # model outputs go to results/{RESULTS_NAME}.*. Change it to keep runs apart.
EXTRA_PIXEL_POINTS = []                   # optional (x, y) UTM points to add to the detailed-polygon output
EXTRA_QOUT_POINTS = []                    # optional (x, y) UTM points to add to the internal-streamflow output

print(f'Run choice: {RUN_CHOICE} | start {START_DATE} | {RUN_HOURS} hours')

## Check the inputs you prepared by hand

The next cell does two things. It **finds the radar folder** and stores it in `RADAR_DIR`, which the weather and rain section needs. It also **reports** whether each hand-made input exists. The report does not stop the notebook: anything marked MISSING needs to be fixed before the section that uses it. (The mesh, soil and weather checks later in the notebook are stricter.)

In [ ]:
# Find the radar folder. The end-labeled set is the one that contains RADAR_LAST_FILE; the older
# start-labeled set does not have that file, so it cannot be picked by mistake.
radar_note = ''
if RADAR_DIR is None:
    root = os.path.expanduser(RADAR_ROOT)
    hits = sorted({os.path.dirname(p) for p in glob.glob(os.path.join(root, '**', RADAR_LAST_FILE), recursive=True)})
    if len(hits) == 1:
        RADAR_DIR = os.path.abspath(hits[0])
    elif len(hits) > 1:
        radar_note = 'more than one folder has ' + RADAR_LAST_FILE + ': ' + ', '.join(hits) + '. Set RADAR_DIR explicitly.'
    else:
        radar_note = f'no folder under {root} contains {RADAR_LAST_FILE}. Set RADAR_DIR explicitly.'
else:
    RADAR_DIR = os.path.abspath(os.path.expanduser(RADAR_DIR))

checks = [
    ('DEM', DEM),
    ('Land cover map', LAND_MAP),
    ('Land cover table', LAND_TABLE),
    ('Weather station list', MET_STATIONS),
]
if RADAR_DIR is not None:
    checks += [('Radar folder', RADAR_DIR),
               ('Radar first file', os.path.join(RADAR_DIR, RADAR_FIRST_FILE)),
               ('Radar last file', os.path.join(RADAR_DIR, RADAR_LAST_FILE))]

for label, path in checks:
    print(f"{'OK     ' if os.path.exists(path) else 'MISSING'}  {label:22s} {path}")
if RADAR_DIR is None:
    print(f'MISSING  {"Radar folder":22s} {radar_note}')
if os.path.basename(os.getcwd()) != 'lab':
    print(f"\nWARNING: the working folder is '{os.getcwd()}'. This notebook expects to run from workspaces/Sycamore/lab/.")

## Project Class: set up the model folders

The `Project` class stores the model name and coordinate system and creates the folder structure tRIBS uses (everything under `data/`, plus `results/` and `restart/`). The first cell below creates it as `proj` and lists the folders; the second shows the stored metadata (name and EPSG code) that every later class receives.

In [ ]:
proj = Project(os.getcwd(), NAME, EPSG)
proj.directories

In [ ]:
proj.meta

## Mesh Class: delineate the watershed and build the mesh

### Delineation
Before a mesh can be made, the DEM has to be turned into a watershed boundary, a stream network and an outlet point. The cell below collects the settings for that (all from the Settings cell) and tells the `Mesh` class where to save the results.

In [ ]:
# Arguments for the delineation step (WhiteboxTools does the work)
preprocess_args = ([OUTLET_X, OUTLET_Y], SNAP_DIS, THRESHOLD_AREA, DEM, VERBOSE_WBT,
                   proj.meta, proj.directories['preprocessing'])

# Where the delineation results are saved (these are the default names, see the pytRIBS documentation)
output_dir = proj.directories['preprocessing']
path_to_raster = f'{output_dir}/{NAME}_clipped_ext.tif'
path_to_watershed = f'{output_dir}/{NAME}_boundary.shp'
path_to_stream_network = f'{output_dir}/{NAME}_stream.shp'
path_to_outlet = f'{output_dir}/{NAME}_outlet.shp'
maxlevel = None          # None = use the largest wavelet level the DEM allows

mesh_generation_args = (path_to_raster, path_to_watershed, path_to_stream_network, path_to_outlet, maxlevel)

### Run the delineation

The next cell runs the delineation and saves the results. A message that says `Could not create output settings.json file` is harmless.

In [ ]:
tmesh = Mesh(preprocess_args=preprocess_args, generate_mesh_args=mesh_generation_args,
             meta=proj.meta, mesh_dir=proj.directories['mesh'])

### Keep only the main basin polygon (required before building the mesh)

The delineated boundary for Sycamore comes out as three pieces: the main basin plus two stray one-cell slivers. `build_mesh` needs a single polygon, so this cell keeps the largest piece.

It changes the boundary held in memory (`tmesh.mesh_generator.watershed`); the shapefile on disk keeps all three pieces. **Rerun this cell after any kernel restart**, right after the `Mesh(...)` cell above.

In [ ]:
gm = tmesh.mesh_generator
geom = unary_union(list(gm.watershed.geometry))
if geom.geom_type == 'MultiPolygon':
    pieces = sorted(geom.geoms, key=lambda g: g.area, reverse=True)
    print(f'Boundary had {len(pieces)} pieces (km2):', [round(p.area / 1e6, 4) for p in pieces])
    geom = pieces[0]
    gm.watershed = gpd.GeoDataFrame(geometry=[geom], crs=gm.watershed.crs)
else:
    print('Boundary is already a single polygon; nothing to remove.')
print(f'Pieces now: {len(gm.watershed)} | area (km2): {gm.watershed.area.sum() / 1e6:.4f}')

### Look at the delineation

#### Figure 1: watershed boundary, stream network and outlet over the DEM

In [ ]:
dem_data = tmesh.mesh_generator.data
valid_data = dem_data[dem_data > -500]
vmin, vmax = np.percentile(valid_data, [2, 98])
print(f'Plotting with elevation limits: {vmin:.1f} m to {vmax:.1f} m')

fig, ax = plt.subplots(figsize=(10, 8))
extent = tmesh.mesh_generator.get_extent()

dem_plot = np.ma.masked_where(~np.isfinite(dem_data) | (dem_data < -500), dem_data)   # no-data cells stay blank
terrain = plt.get_cmap('terrain').copy()
terrain.set_bad('white')
img = ax.imshow(dem_plot, extent=extent, cmap=terrain, vmin=vmin, vmax=vmax)
cbar = fig.colorbar(img, ax=ax, orientation='vertical', shrink=0.8)
cbar.set_label('Elevation (m)', fontsize=14)
cbar.ax.tick_params(labelsize=12)

tmesh.mesh_generator.watershed.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=2, zorder=2)
tmesh.mesh_generator.stream_network.plot(ax=ax, linewidth=1.5, color='blue', zorder=2)
tmesh.mesh_generator.outlet.plot(ax=ax, color='yellow', marker='*', markersize=200, edgecolor='black', zorder=3)

ax.set_xlabel('Easting (UTM)', fontsize=14)
ax.set_ylabel('Northing (UTM)', fontsize=14)
ax.set_title('Watershed, Stream Network, and Outlet', fontsize=16)
ax.tick_params(axis='both', which='major', labelsize=12)

legend_elements = [
    Line2D([0], [0], color='red', lw=2, label='Watershed Boundary'),
    Line2D([0], [0], color='blue', lw=1.5, label='Stream Network'),
    Line2D([0], [0], color='yellow', marker='*', markeredgecolor='black', markersize=15,
           linestyle='None', label='Outlet'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=12, framealpha=0.9)

# Zoom to the watershed with a 1% margin
minx, miny, maxx, maxy = tmesh.mesh_generator.watershed.total_bounds
margin_x, margin_y = (maxx - minx) * 0.01, (maxy - miny) * 0.01
ax.set_xlim(minx - margin_x, maxx + margin_x)
ax.set_ylim(miny - margin_y, maxy + margin_y)

plt.tight_layout()
plt.show()

#### Check the basin area and the streams

A quick check that is worth running on any new basin: the boundary area should be within a few percent of the published drainage area (about 425 km2 for the gauge), and the stream count and length should look reasonable.

In [ ]:
ws = tmesh.mesh_generator.watershed
st = tmesh.mesh_generator.stream_network
print('CRS:', ws.crs)
print('watershed area (km2):', round(ws.area.sum() / 1e6, 4))
print('stream segments:', len(st), '| total length (km):', round(st.length.sum() / 1000, 1))

### Mesh generation

With the boundary, streams and outlet in hand, pytRIBS builds a locally refined mesh in two parts:

1. **Terrain points (Haar wavelet transform).** A wavelet analysis of the DEM picks the points that matter most for describing the terrain. `MESH_THRESHOLD` controls how many are kept: lower values keep more points and make a denser mesh.
2. **Planar Straight Line Graph (PSLG) triangulation.** The boundary and the stream lines are built into the mesh as fixed edges, then the open-source *Triangle* program fills in the triangles between them.

The mesh settings are in the Settings cell. Two things to know:
- `THRESHOLD_AREA` in the delineation step also changes the mesh, because it decides how many streams are embedded in it.
- A dense mesh runs slowly and can crash the kernel while it is being built. The next cell counts how many candidate points the chosen threshold produces **before** anything is built, and stops if the number is above `MAX_CANDIDATE_POINTS`.

In [ ]:
def count_candidates(threshold):
    """Number of terrain points the wavelet step would keep at this threshold."""
    gm = tmesh.mesh_generator
    total = 0
    for level in range(1, gm.maxlevel + 1):
        v = gm.wavelet_packet['v' * level].data
        h = gm.wavelet_packet['h' * level].data
        d = gm.wavelet_packet['d' * level].data
        norm = np.maximum.reduce([np.abs(v), np.abs(h), np.abs(d)]) / gm.normalizing_coeff
        total += np.count_nonzero(norm > 2 ** (-level + 1) * threshold)
    return total


for thr in sorted({0.5, 0.7, 0.9, MESH_THRESHOLD}):
    marker = '  <- MESH_THRESHOLD' if thr == MESH_THRESHOLD else ''
    print(f'threshold {thr}: about {count_candidates(thr)} candidate points{marker}')

n_candidates = count_candidates(MESH_THRESHOLD)
if n_candidates > MAX_CANDIDATE_POINTS:
    raise RuntimeError(f'MESH_THRESHOLD = {MESH_THRESHOLD} gives about {n_candidates} candidate points, above the '
                       f'safety limit of {MAX_CANDIDATE_POINTS}. Raise MESH_THRESHOLD, or raise MAX_CANDIDATE_POINTS '
                       'if you are sure the machine has the memory.')

### Build the mesh

This writes four files, `{NAME}_mesh.nodes`, `.edges`, `.tri` and `.z`, into `data/model/mesh/`. tRIBS reads these four files at the start of a simulation. `diagnostics=True` also writes shapefiles for checking the mesh (triangles, nodes, edges) into `data/preprocessing/`. A long printout follows.

In [ ]:
mesh_prefix = f'{NAME}_mesh'
mesh_base = f"{proj.directories['mesh']}/{mesh_prefix}"
mesh_files = [f'{mesh_base}.{ext}' for ext in ('nodes', 'edges', 'tri', 'z')]

if all(os.path.exists(f) for f in mesh_files):
    # The four mesh files are already on disk, so skip the memory-heavy build and point the model at them.
    tmesh.optmeshinput['value'] = 1           # 1 = read the mesh from files
    tmesh.inputdatafile['value'] = mesh_base  # the file name without .nodes/.edges/.tri/.z
    print('Mesh files found. Skipping the build and reusing:', mesh_base)
else:
    print('Mesh files not found. Building the mesh (this is the memory-heavy step).')
    tmesh.build_mesh(
        method='pslg',
        threshold=MESH_THRESHOLD,
        boundary_buffer_dist=BOUNDARY_BUFFER_DIST,
        boundary_spacing=BOUNDARY_SPACING,
        stream_point_spacing=STREAM_POINT_SPACING,
        stream_clearance_radius=STREAM_CLEAR_RADIUS,
        mesh_quality_opts=QUALITY_OPTS,
        diagnostics=True,                    # write QA shapefiles
        separate_parallel_streams=True,      # second pass that checks the mesh for streams that merge too early
    )

### Check the mesh

Node codes in the mesh files: 0 = interior, 1 = boundary, 2 = outlet, 3 = stream. The counts below should match the printout above. The triangle sizes matter for run time and for how well streambed processes are resolved (questions 9 and 11 for Josh).

In [ ]:
mesh_path = f"{proj.directories['mesh']}/"
mesh_data = InOut.read_mesh_files(mesh_prefix, path=mesh_path)   # nodes (x, y, z), triangles, node codes, edges

code_names = {0: 'interior', 1: 'boundary', 2: 'outlet', 3: 'stream'}
codes_found, code_counts = np.unique(mesh_data['node_codes'], return_counts=True)
print('Nodes by code:', {code_names.get(int(c), int(c)): int(n) for c, n in zip(codes_found, code_counts)},
      '| total:', len(mesh_data['vertices']))

tri = gpd.read_file(f"{proj.directories['preprocessing']}/{mesh_prefix}_triangles.shp")
areas = tri.area
print(f'Triangles: {len(tri)} | median area {areas.median():.0f} m2 | 95th percentile {areas.quantile(0.95):.0f} m2 '
      f'| largest {areas.max():.0f} m2')
print(f'Triangles under 500 m2: {(areas < 500).sum()} ({(areas < 500).mean() * 100:.1f}%) | smallest {areas.min():.1f} m2')

#### Figure 2: triangles around the outlet (zoomed)

In [ ]:
half_width = 1500    # m on each side of the outlet

fig, ax = plt.subplots(figsize=(8, 8))
tri.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.3)
tmesh.mesh_generator.stream_network.plot(ax=ax, color='blue', linewidth=1.5)
tmesh.mesh_generator.outlet.plot(ax=ax, color='yellow', marker='*', markersize=200, edgecolor='black')
ax.set_xlim(OUTLET_X - half_width, OUTLET_X + half_width)
ax.set_ylim(OUTLET_Y - half_width, OUTLET_Y + half_width)
ax.set_title('Mesh triangles near the outlet')
plt.show()

#### Figure 3: triangle size across the basin (log scale)

In [ ]:
tri['log10_area'] = np.log10(tri.area)

fig, ax = plt.subplots(figsize=(8, 10))
tri.plot(ax=ax, column='log10_area', cmap='viridis', edgecolor='none',
         legend=True, legend_kwds={'label': 'log10 triangle area (m2)', 'shrink': 0.7})
tmesh.mesh_generator.stream_network.plot(ax=ax, color='red', linewidth=0.6)
ax.set_title('Triangle size (log10 m2) with streams in red')
plt.show()

### After the first run: check the Voronoi polygons

tRIBS writes the Voronoi polygon of every mesh node only when it runs, so the check of this mesh's polygons cannot be done here. It is the last section of `Compare_Sycamore_Runs.ipynb`; use it after a run.

## Soil Class: build the soil inputs and wire them into the model

tRIBS can take soil properties in two ways: a soil ID map plus a lookup table, or **gridded parameter rasters**. SMF used the first, with ADOT soil polygons. Sycamore uses the second ("gridded mode", `OPTSOILTYPE = 1`) with **SOLUS**, a 100 m national soil grid.

What happens in this section:
1. Download SOLUS sand, silt, clay and bulk density at three depths, plus depth to bedrock (skipped if the files are already in `../init_data/solus`).
2. `run_soil_workflow` turns texture into tRIBS soil properties with the Rosetta pedotransfer model: saturated conductivity `Ks`, `thetaS`, `thetaR`, `m` and `PsiB`, plus the conductivity decay parameter `f`. Each one is written as its own grid. It also writes a texture-class map and table, and `scgrid.gdf`, the file that tells tRIBS which grid is which.
3. Fill the leftover constants in the soil table, and make the initial groundwater grid.
4. Copy everything tRIBS reads into `data/model/soil/` and point the `Soil` object at the copies.

Treat the SOLUS `f` as a starting value only: it comes from a fit over just the top 30 cm.

In [ ]:
soil = Soil(meta=proj.meta)

# Outline of the basin as drawn by the delineation step (the file keeps all three pieces). Used for the checks below.
basin_shapes = list(gpd.read_file(path_to_watershed).geometry)

### Helper functions for reading rasters inside the basin

Two small functions used by the soil checks below. `values_in_basin` cuts a raster to the basin outline; `valid_basin_values` returns just the usable cell values inside the basin as one flat list, which is what the minimum / median / maximum printouts use. Run the cell once; nothing is drawn.

In [ ]:
def values_in_basin(path, shapes):
    """Return (values, is_inside, nodata) for a raster, cropped to the basin's bounding box.

    is_inside is True for cells inside the basin outline."""
    with rasterio.open(path) as r:
        out, _ = mask(r, shapes, crop=True, filled=False)
        values = np.ma.getdata(out[0])
        is_inside = ~np.ma.getmaskarray(out[0])
        return values, is_inside, r.nodata


def valid_basin_values(path, shapes):
    """Return a 1-D array of the raster's values inside the basin, with no-data cells dropped."""
    values, is_inside, nodata = values_in_basin(path, shapes)
    vals = values[is_inside]
    return vals[vals != nodata] if nodata is not None else vals

### 1. Download SOLUS (only what is missing)

The download box is the whole DEM rectangle, which includes the 5 km buffer around the basin. `get_solus_grids` skips files that already exist, and the bedrock download is skipped if `bedrock_depth_m.asc` is already there.

In [ ]:
with rasterio.open(DEM) as dem_src:
    b = dem_src.bounds
bbox = [b.left, b.bottom, b.right, b.top]          # [minx, miny, maxx, maxy] in EPSG 26912
print('Download box (EPSG 26912):', bbox)

# pytRIBS writes into a folder called "solus" in the current folder, so step into SOLUS_DIR for the download
start_dir = os.getcwd()
os.chdir(SOLUS_DIR)
try:
    solus_files = soil.get_solus_grids(bbox, depths=SOLUS_DEPTHS, variables=SOLUS_VARIABLES)
finally:
    os.chdir(start_dir)                            # always come back, even if the download fails

# Depth to bedrock, converted from cm to m
bedrock_path = f'{SOLUS_DIR}/bedrock_depth_m.asc'
if os.path.exists(bedrock_path):
    print('Bedrock file already exists, not downloaded again:', bedrock_path)
else:
    bedrock_path = soil.get_solus_bedrock(bbox, output_dir=SOLUS_DIR)
    if bedrock_path is None:
        raise RuntimeError('The SOLUS bedrock download failed (see the message above).')
print('Bedrock file:', bedrock_path)

### 1b. Deepen the bedrock grid (optional test)

Josh's test on SMF was to add 1 m to every depth-to-bedrock cell. `BEDROCK_ADD_M` in Settings sets how much to add (0 = use the SOLUS values as they are). The deepened grid is saved under its own name (for example `bedrock_depth_plus1m.asc`) and the SOLUS file itself is never changed. The variable `bedrock_path`, set here, is the bedrock file the rest of the notebook uses.

In [ ]:
# Deepen the bedrock grid (Josh's test: SMF got +1 m everywhere). BEDROCK_ADD_M is set in Settings; 0 = no change.
original_bedrock = f'{SOLUS_DIR}/bedrock_depth_m.asc'          # the SOLUS file itself, never changed
if BEDROCK_ADD_M == 0:
    bedrock_path = original_bedrock
else:
    bedrock_path = f'{SOLUS_DIR}/bedrock_depth_plus{BEDROCK_ADD_M:g}m.asc'
    grid = soil._read_ascii(original_bedrock)
    nodata = grid['profile'].get('nodata', -9999)
    depth = grid['data'].astype('float32')
    depth = np.where(depth == nodata, nodata, depth + BEDROCK_ADD_M)    # no-data cells stay no-data
    soil._write_ascii({'data': depth, 'profile': grid['profile']}, bedrock_path, decimals=2)
    print(f'Added {BEDROCK_ADD_M:g} m to every bedrock cell that has data.')
print('Bedrock file in use:', bedrock_path)

### 2. First look at the downloaded grids

SOLUS marks rock outcrop and water with zeros, so a large share of zeros in the corners (outside the basin) is expected. What matters is the next check.

In [ ]:
print('--- Texture and bulk density rasters ---')
for fname in solus_files:
    path = f'{SOLUS_DIR}/solus/{fname}'
    if not os.path.exists(path):
        print(f'{fname}: MISSING (the download failed for this one)')
        continue
    with rasterio.open(path) as r:
        a = r.read(1, masked=True)                       # cells flagged as no-data are left out of min and max
        print(f'{fname}: {a.shape[0]} rows x {a.shape[1]} cols, cell {r.res[0]:.0f} x {r.res[1]:.0f} m, '
              f'min {a.min():.1f}, max {a.max():.1f}, no data: {np.ma.getmaskarray(a).mean() * 100:.1f}%, '
              f'cells equal to 0: {(a == 0).sum() / a.size * 100:.1f}%')

print('\n--- Depth to bedrock (m) ---')
with rasterio.open(bedrock_path) as r:
    a = r.read(1, masked=True)
    print(f'{a.shape[0]} rows x {a.shape[1]} cols, cell {r.res[0]:.0f} x {r.res[1]:.0f} m, nodata value: {r.nodata}')
    print(f'cells with no data: {np.ma.getmaskarray(a).mean() * 100:.1f}%')
    print(f'min {a.min():.2f} m, median {np.ma.median(a):.2f} m, max {a.max():.2f} m')
    print(f'cells at exactly 0 m: {(a == 0).sum() / a.count() * 100:.1f}%')

### 3. Check the grids inside the basin outline

Inside the basin the share of no-data and non-soil cells should be about 0%. The maps show the whole downloaded box with the basin outline in red (gray = zero or no data).

In [ ]:
print('--- Inside the basin outline only ---')
layers = {}
for var in ['sandtotal', 'silttotal', 'claytotal']:
    v, is_inside, nd = values_in_basin(f'{SOLUS_DIR}/solus/{var}_0-5cm_mean.tif', basin_shapes)
    layers[var] = (v.astype('int16'), nd)
n_inside = is_inside.sum()
nodata_cells = np.zeros_like(is_inside)
for var, (v, nd) in layers.items():
    nodata_cells |= (v == nd)
total = sum(v for v, nd in layers.values())
non_soil = is_inside & ~nodata_cells & (total == 0)
odd = is_inside & ~nodata_cells & (total > 0) & ((total < 95) | (total > 105))
print(f'cells inside basin: {n_inside}')
print(f'  flagged no-data by SOLUS: {(is_inside & nodata_cells).sum() / n_inside * 100:.1f}%')
print(f'  non-soil (sand + silt + clay = 0, e.g. rock outcrop or water): {non_soil.sum() / n_inside * 100:.1f}%')
print(f'  other cells where sand + silt + clay is not near 100: {odd.sum() / n_inside * 100:.1f}%')

bed, bed_inside, bed_nd = values_in_basin(bedrock_path, basin_shapes)
bed_valid = bed_inside & (bed != bed_nd)
print(f'bedrock depth inside basin: no data {(bed_inside & (bed == bed_nd)).sum() / n_inside * 100:.1f}%, '
      f'exactly 0 m {(bed_valid & (bed == 0)).sum() / n_inside * 100:.1f}%, '
      f'min {bed[bed_valid].min():.2f} m, median {np.median(bed[bed_valid]):.2f} m, max {bed[bed_valid].max():.2f} m')

# Maps of the whole downloaded grids with the basin outline on top
basin_gdf = gpd.read_file(path_to_watershed)
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, path, title in [(axes[0], f'{SOLUS_DIR}/solus/sandtotal_0-5cm_mean.tif', 'Sand, 0-5 cm (%)'),
                        (axes[1], bedrock_path, 'Depth to bedrock (m)')]:
    with rasterio.open(path) as r:
        a = r.read(1).astype('float32')
        nd = r.nodata
        ext = (r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top)
    a = np.ma.masked_where((a == 0) | (a == nd), a)          # zeros and no-data show as gray
    cmap = plt.get_cmap('viridis').copy()
    cmap.set_bad('lightgray')
    im = ax.imshow(a, extent=ext, cmap=cmap)
    basin_gdf.boundary.plot(ax=ax, color='red', linewidth=1)
    ax.set_title(title + '\n(gray = zero or no data; red = basin)')
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

### 4. Run the soil workflow (only if its files are missing)

This is the slow step (several minutes): it fills gaps, runs Rosetta, fits `f`, and writes the grids, the class map, `soils.sdt` and `scgrid.gdf` into `../init_data`. It is skipped when all of those files already exist; set `REBUILD_SOIL = True` in Settings to force it.

pytRIBS 1.0.0 quirks handled here: the workflow wants a plain shapely `box` (a GeoDataFrame fails with `KeyError: 0`), and it changes the working folder without changing back if it fails.

In [ ]:
soil_outputs = [f'{SOLUS_DIR}/scgrid.gdf', f'{SOLUS_DIR}/soils.sdt', f'{SOLUS_DIR}/soils_textures.csv',
                f'{SOLUS_DIR}/solus/soil_classes.soi', f'{SOLUS_DIR}/solus/Ks_0-5cm.asc', f'{SOLUS_DIR}/solus/f.asc']
missing_outputs = [p for p in soil_outputs if not os.path.exists(p)]

if REBUILD_SOIL or missing_outputs:
    if missing_outputs and not REBUILD_SOIL:
        print('Missing, so the workflow will run:', missing_outputs)
    watershed_box = box(*bbox)
    start_dir = os.getcwd()
    try:
        soil.run_soil_workflow(watershed_box, SOLUS_DIR, source='SOLUS')
    finally:
        os.chdir(start_dir)                    # the workflow does not change back if it fails
    print('Back in:', os.getcwd())
else:
    print('All soil workflow outputs already exist; skipped. Set REBUILD_SOIL = True to run it again.')

# Point the Soil object at the gridded soil files explicitly (the workflow also does this, but a skipped run does not)
soil.optsoiltype['value'] = 1
soil.scgrid['value'] = f'{SOLUS_DIR}/scgrid.gdf'
soil.soiltablename['value'] = f'{SOLUS_DIR}/soils.sdt'
soil.soilmapname['value'] = f'{SOLUS_DIR}/solus/soil_classes.soi'

### 5. Look at the results

`scgrid.gdf` has one row per grid tRIBS reads: `KS` (saturated conductivity), `TR` (thetaR), `TS` (thetaS), `PB` (PsiB), `PI` (pore-size index `m`), `FD` (decay `f`) and `PO` (porosity, set equal to thetaS).

In [ ]:
print('Soil settings now:')
print('  optsoiltype :', soil.optsoiltype['value'])
print('  scgrid      :', soil.scgrid['value'])
print('  soil table  :', soil.soiltablename['value'])
print('  soil map    :', soil.soilmapname['value'])

print('\nGrids and maps in the SOLUS folder:')
for fname in sorted(os.listdir(f'{SOLUS_DIR}/solus')):
    if fname.endswith(('.asc', '.soi')):
        print('  ', fname)

print('\nscgrid.gdf says:')
print(open(soil.scgrid['value']).read())

# Texture classes inside the basin
tex = {}
with open(f'{SOLUS_DIR}/soils_textures.csv') as fh:
    next(fh)
    for line in fh:
        key, value = line.strip().split(',', 1)
        tex[int(key)] = value
with rasterio.open(soil.soilmapname['value']) as r:
    out, _ = mask(r, basin_shapes, crop=True, filled=False)
is_inside = ~np.ma.getmaskarray(out[0])
ids, counts = np.unique(np.ma.getdata(out[0])[is_inside], return_counts=True)
print('Soil texture classes inside the basin:')
for class_id, n in zip(ids, counts):
    print(f'  class {class_id} ({tex.get(int(class_id), "?")}): {n / counts.sum() * 100:.1f}% of basin cells')

# Value ranges inside the basin for each parameter grid
print('\nInside the basin:')
for gname in ['Ks_0-5cm', 'Ks_5-15cm', 'Ks_15-30cm', 'theta_s_0-5cm', 'theta_r_0-5cm', 'm_0-5cm', 'psib_0-5cm', 'f']:
    vals = valid_basin_values(f'{SOLUS_DIR}/solus/{gname}.asc', basin_shapes)
    print(f'  {gname:14s} min {vals.min():10.4g}   median {np.median(vals):10.4g}   max {vals.max():10.4g}')

#### Figure 4: the two soil parameters most likely to be calibrated, `Ks` (0-5 cm) and `f`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
minx, miny, maxx, maxy = basin_gdf.total_bounds
for ax, gname, title in [(axes[0], 'Ks_0-5cm', 'Saturated conductivity Ks, 0-5 cm (mm/hr)'),
                         (axes[1], 'f', 'Conductivity decay f (1/mm)')]:
    with rasterio.open(f'{SOLUS_DIR}/solus/{gname}.asc') as r:
        a = r.read(1, masked=True).astype('float32')
        ext = (r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top)
    lo, hi = np.percentile(valid_basin_values(f'{SOLUS_DIR}/solus/{gname}.asc', basin_shapes), [2, 98])
    im = ax.imshow(a, extent=ext, cmap='viridis', vmin=lo, vmax=hi)
    basin_gdf.boundary.plot(ax=ax, color='red', linewidth=1)
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

### 6. Fill the leftover soil table constants

In gridded mode the grids supply Ks, thetaS, thetaR, m, PsiB, f and porosity, and the 9999.99 placeholders in `soils.sdt` are ignored (read from the tRIBS source; the first model run will confirm). The columns no grid supplies are the anisotropy ratios `As` and `Au`, the volumetric heat conductivity `ks` and the soil heat capacity `Cs`. They come from `SOIL_CONSTANTS` in Settings. Running this cell twice is harmless.

In [ ]:
soil_table = soil.read_soil_table(textures=True)          # reads soils.sdt (and the soils_textures.csv next to it)

for cls in soil_table:
    cls.update(SOIL_CONSTANTS)

soil.write_soil_table(soil_table, soil.soiltablename['value'], textures=True)
print(open(soil.soiltablename['value']).read())

### 7. Initial groundwater

tRIBS needs a starting water table. For these shallow soils over bedrock in a dry climate we assume it starts close to bedrock: the depth to the water table is `GW_FRACTION` (95%) of the depth to bedrock, so if bedrock is 1 m down the water table starts at 0.95 m below the surface. **This is an assumption to test** (the spin-up question, number 10 for Josh). The grid is in mm below the surface.

The cell also lists the `f` values, because 9% of cells sit at the lower limit of 0.0001 where the fit found no usable decay.

In [ ]:
gw_name = f'InitGW_{int(round(GW_FRACTION * 100))}pct_plus{BEDROCK_ADD_M:g}m_mm.asc'
gw_path = soil.generate_initial_groundwater(bedrock_path, GW_FRACTION, filename=f'{SOLUS_DIR}/{gw_name}')
print('Made from bedrock file:', bedrock_path)

gw = valid_basin_values(gw_path, basin_shapes)
print(f'Initial water table depth (mm): min {gw.min():.0f}, median {np.median(gw):.0f}, max {gw.max():.0f}')
f_vals = valid_basin_values(f'{SOLUS_DIR}/solus/f.asc', basin_shapes)
print(f'f (1/mm): 10th percentile {np.percentile(f_vals, 10):.5f}, median {np.median(f_vals):.5f}, '
      f'90th percentile {np.percentile(f_vals, 90):.5f}')
print(f'cells where f sits at the 0.0001 floor: {(f_vals <= 1.0001e-4).mean() * 100:.1f}%')

### 8. Copy the soil files into the model folder and point the Soil object at them

tRIBS reads its inputs from `data/model/`, so this cell copies in:
- the grids listed in `scgrid.gdf` (a model-folder copy of `scgrid.gdf` is written with the new paths),
- the soil table (`soils.sdt`) and the texture-class map (`soil_classes.soi`, which tells tRIBS which table row applies where),
- the bedrock depth grid and the initial groundwater grid.

**The three calibration multipliers are applied here.** The `KS` grid is multiplied by `KS_MULT`, the `PB` grid (PsiB) by `PSIB_MULT` and the `FD` grid (f) by `F_MULT`; a multiplier of 1 gives a plain copy. A multiplied grid is saved with a name that ends in the multiplier (for example `_x0.2`), and the SOLUS files are never changed. After f is multiplied, no cell is allowed below 0.0001, because tRIBS divides by f. So **rerun this cell whenever you change a multiplier**, and read the three lines at the bottom of its output (Ks, PsiB and f medians, SOLUS against the value in use): they confirm which multipliers this run uses.

Then it sets the `Soil` options: `optsoiltype = 1` (gridded soils), `optbedrock` and `optgwfile = 0` (read the groundwater grid).

In [ ]:
soil_dir = proj.directories['soil']                        # data/model/soil

# Read the grid list that run_soil_workflow wrote, copy each grid, and write a model-folder scgrid.gdf.
# Three grids can be multiplied on the way in (the SOLUS files themselves are never changed):
#   variable KS (surface saturated conductivity)      by KS_MULT
#   variable PB (air-entry pressure PsiB)             by PSIB_MULT
#   variable FD (conductivity decay with depth, f)    by F_MULT
# A multiplier of 1 means the grid is simply copied. PsiB values are negative numbers, so a PSIB_MULT of 0.5 halves the
# size of the air-entry pressure (for example -93.7 mm becomes -46.9 mm).
#
# f has a floor. tRIBS divides by f in many places, and pytRIBS keeps f at or above 1e-4 (the min_f option of
# compute_ks_decay) because smaller values cannot be handled numerically. So after f is multiplied, no cell is allowed
# to fall below F_FLOOR. This means cells that already sit at the floor do not change when F_MULT is below 1.
# f is also a small number (about 0.0002 to 0.007), so its scaled grid is written with 8 decimals instead of 4.
F_FLOOR = 1e-4
MULTIPLIERS = {'KS': KS_MULT, 'PB': PSIB_MULT, 'FD': F_MULT}
FLOORS = {'FD': F_FLOOR}                                   # smallest value allowed after scaling
DECIMALS = {'FD': 8}                                       # decimals written for a scaled grid (4 if not listed)

with open(f'{SOLUS_DIR}/scgrid.gdf') as fh:
    grid_rows = list(csv.DictReader(fh))

new_params = []
used_names = {}                                            # variable -> grid file name in use for this run
source_files = {}                                          # variable -> the SOLUS grid file it came from
for row in grid_rows:
    variable = row['Variable'].strip()
    base, ext = row['BasePath'].strip(), row['FileExtension'].strip()
    name = os.path.basename(base)
    source_files[variable] = f'{base}.{ext}'
    mult = MULTIPLIERS.get(variable, 1)
    if mult != 1:
        grid = soil._read_ascii(f'{base}.{ext}')                     # the SOLUS file itself is never changed
        nodata = grid['profile'].get('nodata', -9999)
        data = grid['data'].astype('float32')
        scaled = data * mult
        if variable in FLOORS:
            scaled = np.maximum(scaled, FLOORS[variable])            # keep f at or above the floor
        scaled = np.where(data == nodata, nodata, scaled)            # no-data cells stay no-data
        name = f'{name}_x{mult:g}'
        soil._write_ascii({'data': scaled, 'profile': grid['profile']}, f'{soil_dir}/{name}.{ext}',
                          decimals=DECIMALS.get(variable, 4))
    else:
        shutil.copy(f'{base}.{ext}', soil_dir)
    used_names[variable] = name
    new_params.append({'Variable Name': variable,
                       'Raster Path': f'{soil_dir}/{name}',
                       'Raster Extension': ext})
InOut.write_grid_data_file(f'{soil_dir}/scgrid.gdf', {'Parameters': new_params})

# Soil table, texture-class map, bedrock depth and initial groundwater
shutil.copy(f'{SOLUS_DIR}/soils.sdt', soil_dir)
shutil.copy(f'{SOLUS_DIR}/solus/soil_classes.soi', soil_dir)
shutil.copy(bedrock_path, soil_dir)
shutil.copy(gw_path, soil_dir)

# Point the Soil object at the copies in the model folder
soil.optsoiltype['value'] = 1
soil.scgrid['value'] = f'{soil_dir}/scgrid.gdf'
soil.soiltablename['value'] = f'{soil_dir}/soils.sdt'
soil.soilmapname['value'] = f'{soil_dir}/soil_classes.soi'
soil.optbedrock['value'] = 1                               # 1 = use the bedrock depth grid
soil.bedrockfile['value'] = f'{soil_dir}/{os.path.basename(bedrock_path)}'
soil.optgwfile['value'] = 0                                # 0 = resample the ASCII grid in GWATERFILE
soil.gwaterfile['value'] = f'{soil_dir}/{os.path.basename(gw_path)}'

print('Files now in', soil_dir)
for fname in sorted(os.listdir(soil_dir)):
    print('  ', fname)
print('\nmodel-folder scgrid.gdf:')
print(open(soil.scgrid['value']).read())

ks_name, pb_name, fd_name = used_names['KS'], used_names['PB'], used_names['FD']
ks_solus = valid_basin_values(source_files['KS'], basin_shapes)
ks_used = valid_basin_values(f'{soil_dir}/{ks_name}.asc', basin_shapes)
print(f'Ks 0-5 cm inside the basin (mm/hr): SOLUS median {np.median(ks_solus):.2f} | in use for this run {np.median(ks_used):.2f} '
      f'(KS_MULT = {KS_MULT:g}, grid file {ks_name}.asc)')
pb_solus = valid_basin_values(source_files['PB'], basin_shapes)
pb_used = valid_basin_values(f'{soil_dir}/{pb_name}.asc', basin_shapes)
print(f'PsiB 0-5 cm inside the basin (mm, negative): SOLUS median {np.median(pb_solus):.2f} | in use for this run '
      f'{np.median(pb_used):.2f} (PSIB_MULT = {PSIB_MULT:g}, grid file {pb_name}.asc)')
f_solus = valid_basin_values(source_files['FD'], basin_shapes)
f_used = valid_basin_values(f'{soil_dir}/{fd_name}.asc', basin_shapes)
print(f'f inside the basin (1/mm): SOLUS median {np.median(f_solus):.5f}, {(f_solus <= 1.0001e-4).mean() * 100:.1f}% of cells at the floor | '
      f'in use for this run median {np.median(f_used):.5f}, {(f_used <= 1.0001e-4).mean() * 100:.1f}% at the floor '
      f'(F_MULT = {F_MULT:g}, grid file {fd_name}.asc)')

## Met Class: weather stations and radar rainfall

tRIBS needs two kinds of forcing, and Sycamore gets them from different places:

- **Weather** (air temperature, humidity, wind, solar radiation, pressure) comes from three NLDAS-2 **point stations** at 657, 1,247 and 1,435 m, made by `Generate_Met_Forcing_Sycamore.ipynb`. The station list is `Master_Met.sdf`; each station has its own `.mdf` data file. Air temperature is adjusted to each node's elevation with `TEMPLAPSE` (-0.0065 K per m by default); pressure, humidity, wind and radiation are not adjusted (question 4 for Josh).
- **Rain** comes from **hourly radar grids** (MRMS, gauge-corrected, 1 km). A file is labeled by the **end** of its hour: `radar_1001201601.asc` is the rain that fell from 00:00 to 01:00 on 10/01/2016, and the hour ending at midnight is written as hour `00` of the next day. The NLDAS `.mdf` files carry no rain, and there is no rain-gauge file (`RAINSOURCE = 1` means gridded rain, so tRIBS never opens one).

The weather rows may also need the "end of hour" label; that is question 12 for Josh and does not change this notebook.

In [ ]:
if RADAR_DIR is None:
    raise FileNotFoundError('No radar folder was found. ' + radar_note)

met = Met(meta=proj.meta)

# Weather: point stations
met.metdataoption['value'] = 1                    # 1 = weather station point data
met.hydrometstations['value'] = MET_STATIONS      # station list (*.sdf)

# Rain: gridded radar. RAINFILE is the folder plus the start of the file name; tRIBS adds MMDDYYYYHH and the extension.
met.rainsource['value'] = 1                       # 1 = gridded rainfall, 2 = rain gauges
met.rainfile['value'] = f'{RADAR_DIR}/{RADAR_BASENAME}'
met.rainextension['value'] = RADAR_EXT
met.gaugestations['value'] = None                 # no rain-gauge file is used

print('Weather stations:', met.hydrometstations['value'])
print('Radar rain      :', met.rainfile['value'] + '*.' + met.rainextension['value'])

### Check the weather files against the run window

For each station this reads the `.mdf` file and compares its first and last hour with the run window. The weather files end at 09/30/2017 23:00, so a **full-year** run (which ends 10/01/2017 00:00) goes one hour past them. If tRIBS complains on the full-year run, shorten `RUN_HOURS` for that run to 8759.

In [ ]:
run_start = datetime.strptime(START_DATE, '%m/%d/%Y/%H/%M')
run_end = run_start + timedelta(hours=RUN_HOURS)
print(f'Run window: {run_start:%Y-%m-%d %H:%M} to {run_end:%Y-%m-%d %H:%M} ({RUN_HOURS} hours)\n')

stations = met.read_met_sdf()
if not stations:
    raise RuntimeError(f'No stations were read from {MET_STATIONS}.')

for s in stations:
    path = s['file_path']
    if not os.path.exists(path):
        print(f"MISSING  station {s['station_id']}: {path}")
        continue
    df = met.read_met_station(path)
    first, last = df['date'].min(), df['date'].max()
    steps = df['date'].diff().dropna().unique()
    hourly = len(steps) == 1 and steps[0] == np.timedelta64(1, 'h')
    covers_start = first <= run_start
    covers_end = last >= run_end
    note = []
    if not hourly:
        note.append('NOT a regular hourly series')
    if not covers_start:
        note.append('starts after the run start')
    if not covers_end:
        note.append(f'ends {(run_end - last).total_seconds() / 3600:.0f} h before the run ends')
    print(f"{'OK     ' if not note else 'CHECK  '}station {s['station_id']}: {len(df)} rows, {first:%Y-%m-%d %H:%M} to {last:%Y-%m-%d %H:%M}, "
          f"elev {s['elevation']:.0f} m | {path}" + (' | ' + '; '.join(note) if note else ''))

### Check the radar files against the run window

For every hour of the run, tRIBS needs the file labeled with the **end** of that hour. A missing file only produces a warning in tRIBS, which then searches forward for the next file, so it is better to find gaps here. (The 17 hours that were missing from the source archive were written as zero-rain grids, so no file should be missing.)

In [ ]:
on_disk = set(os.listdir(RADAR_DIR))
needed = [f"{RADAR_BASENAME}{(run_start + timedelta(hours=h)):%m%d%Y%H}.{RADAR_EXT}" for h in range(1, RUN_HOURS + 1)]
missing = [n for n in needed if n not in on_disk]

n_radar = sum(1 for n in on_disk if n.startswith(RADAR_BASENAME) and n.endswith('.' + RADAR_EXT))
print(f'{n_radar} radar files in {RADAR_DIR}')
print(f'This run needs {len(needed)} files: first {needed[0]}, last {needed[-1]}')
if missing:
    print(f'MISSING {len(missing)} files, for example: {missing[:5]}')
else:
    print('OK: every hourly file the run needs is there.')

## Land Class: land cover map and parameter table

Land cover is the LANDFIRE 2016 Remap existing vegetation type, grouped into 8 classes. Both inputs were made outside this notebook, so this section only copies them into the model folder and checks them:

- `LandUse.asc` (10 m class map) from `make_sycamore_landuse_map.py`.
- `land_use_params.ldt` (one row per class: interception, albedo, vegetation height, cover fraction, leaf area index, stress thresholds, and so on) from `make_sycamore_landuse_table.py`. Vegetation cover (`V`) and height (`h`) are LANDFIRE averages by class; the other columns are Josh's South Mountain values. The table also carries both soil-moisture stress thresholds, which must lie between the largest `thetaR` and the smallest `thetaS` near the basin (that script checked this).

If a class changes, change it in those scripts and rerun them, not in this notebook.

In [ ]:
land = Land(meta=proj.meta)

land_dir = proj.directories['land']
for src in (LAND_MAP, LAND_TABLE):
    shutil.copy(src, land_dir)

land.landmapname['value'] = f'{land_dir}/{os.path.basename(LAND_MAP)}'
land.landtablename['value'] = f'{land_dir}/{os.path.basename(LAND_TABLE)}'
land.optlanduse['value'] = 0               # 0 = class map plus lookup table

print('Land cover map  :', land.landmapname['value'])
print('Land cover table:', land.landtablename['value'])

### Land cover table and class check

The table, and a check that every class in the map has a row in the table (and the other way round):

In [ ]:
land_table = pd.DataFrame(land.read_landuse_table())
land_table['ID'] = land_table['ID'].astype(int)
land_table.insert(1, 'Class', land_table['ID'].map(LAND_CLASS_NAMES))
display(land_table)

with rasterio.open(land.landmapname['value']) as r:
    land_grid = r.read(1)
    land_nodata = r.nodata
    land_extent = (r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top)
map_ids = set(int(i) for i in np.unique(land_grid) if i != land_nodata)
table_ids = set(land_table['ID'])
print('Classes in the map  :', sorted(map_ids))
print('Classes in the table:', sorted(table_ids))
if map_ids - table_ids:
    print('PROBLEM: classes in the map with no table row:', sorted(map_ids - table_ids))
if table_ids - map_ids:
    print('Note: table rows with no cells in the map:', sorted(table_ids - map_ids))
if map_ids <= table_ids:
    print('OK: every class in the map has a row in the table.')

#### Figure 5: land cover classes

In [ ]:
classes = sorted(map_ids)
cmap = plt.get_cmap('tab10', len(classes))
step = 4                                          # draw every 4th cell (40 m) so the 10 m map plots quickly
small = land_grid[::step, ::step]
index_grid = np.full(small.shape, np.nan)
for k, class_id in enumerate(classes):
    index_grid[small == class_id] = k

fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(index_grid, extent=land_extent, cmap=cmap, vmin=-0.5, vmax=len(classes) - 0.5, interpolation='nearest')
basin_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5)
minx, miny, maxx, maxy = basin_gdf.total_bounds
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_xlabel('Easting (UTM)')
ax.set_ylabel('Northing (UTM)')
ax.set_title('Land cover classes')
ax.legend(handles=[Patch(facecolor=cmap(k), label=f'{c}  {LAND_CLASS_NAMES.get(c, "")}') for k, c in enumerate(classes)],
          loc='upper left', bbox_to_anchor=(1.01, 1.0), fontsize=9, framealpha=0.9)      # legend beside the map, not on it
plt.tight_layout()
plt.show()

## Model Class: options, output nodes, checks, and the input file

The `Model` class gathers everything built above (mesh, soil, land, weather and rain) and holds the remaining tRIBS options. Because the four objects are shared with the `Model`, a setting made on `soil`, `land` or `met` is already part of `model`.

This section is the one to re-run when you change `RUN_CHOICE`. If you changed `KS_MULT`, `PSIB_MULT` or `F_MULT`, rerun the soil copy cell (step 8 of the Soil section) first, because that is where the multipliers are applied.

In [ ]:
model = Model(met=met, land=land, soil=soil, mesh=tmesh, meta=proj.meta)

### Channel transmission losses and routing

Channel losses use the **constant loss** method, which has one parameter:
- `CHANNELCONDUCTIVITY` is the steady-state streambed conductivity in mm/hr. It is the same for every reach in the model.

Set `OPTPERCOLATION = 0` in Settings to run with no channel losses at all.

The kinematic routing parameters:
- `KINEMVELCOEF` is a direct multiplier on hillslope velocity; doubling it halves the hillslope travel time. (3 gives about 0.03 m/s, 10 about 0.09 m/s, 50 about 0.45 m/s.)
- `FLOWEXP` controls how much velocity scales with discharge; lower values mean more constant velocity regardless of storm size.
- `CHANNELROUGHNESS` is Manning's n for the channels.
- `CHANNELWIDTHCOEFF` is the coefficient in the channel width-area relationship; a smaller value makes flow move faster through the channel (usual range 0.5 to 2.5).

These started as SMF's values and are calibration choices now (see the Settings notes); they are not measured Sycamore values.

In [ ]:
# Channel transmission losses
model.optpercolation['value'] = OPTPERCOLATION
model.channelconductivity['value'] = CHANNELCONDUCTIVITY

# Kinematic routing
model.kinemvelcoef['value'] = KINEMVELCOEF
model.flowexp['value'] = FLOWEXP
model.channelroughness['value'] = CHANNELROUGHNESS
model.channelwidthcoeff['value'] = CHANNELWIDTHCOEFF

### Run options, time, rain interval and solar position

- The mesh options were set by `build_mesh` (`OPTMESHINPUT = 1`, mesh base name) and the soil and land options in their sections above; the cell prints them as a check.
- `UTCOFFSET`, `CENTROIDLAT` and `CENTROIDLONG` are used by tRIBS for solar position whenever evapotranspiration is on. The centroid was filled in by the delineation step; `UTCOFFSET` is set from Settings (Arizona does not use daylight saving).
- `STARTDATE` and `RUNTIME` (hours) come from the run you picked in Settings.
- If `OPTSNOW = 1`, the second cell below also writes the snow parameter file.

In [ ]:
# Run mode
model.parallelmode['value'] = 0                  # 0 = serial
model.optsnow['value'] = OPTSNOW                 # 0 = snow module off
model.optlanduse['value'] = 0                    # 0 = class map plus lookup table
model.optbedrock['value'] = 1                    # 1 = bedrock depth grid

# Simulation time
model.startdate['value'] = START_DATE
model.runtime['value'] = RUN_HOURS
model.rainintrvl['value'] = RAIN_INTERVAL_HR     # hours between radar files
model.outfilename['value'] = f"{proj.directories['results']}/{RESULTS_NAME}"

# Solar position
model.utcoffset['value'] = UTC_OFFSET

print('Options now set on the model:')
for opt in ['optmeshinput', 'inputdatafile', 'optsoiltype', 'optbedrock', 'optgwfile', 'optlanduse', 'metdataoption',
            'rainsource', 'rainintrvl', 'optsnow', 'optpercolation', 'channelconductivity', 'startdate', 'runtime',
            'outfilename', 'utcoffset', 'centroidlat', 'centroidlong', 'templapse']:
    print(f'  {opt:20s}', model.options[opt]['value'])

if model.centroidlat['value'] is None or model.centroidlong['value'] is None:
    raise RuntimeError('CENTROIDLAT / CENTROIDLONG are empty. They are filled in when Mesh(...) delineates the basin; '
                       'rerun that cell, or set them with model.centroidlat["value"] = ... and model.centroidlong["value"] = ...')

In [ ]:
# Snow settings file (only used when OPTSNOW = 1)
if OPTSNOW == 1:
    model.write_snow_params('data/model/snow_params.spf')
    print('SNOWFILENAME =', model.options['snowfilename']['value'])

### Output nodes

tRIBS writes full results for every Voronoi polygon, but it can only write the detailed time series for a few. Two lists are needed:

- **Pixel nodes** (`NODEOUTPUTLIST`): detailed time series for individual Voronoi polygons (the `.pixel` files).
- **Internal streamflow nodes** (`OUTLETNODELIST`): discharge at stream nodes inside the basin (the `.qout` files). These must be stream nodes.

The node ID is the node's row number in the mesh `.nodes` file, counting from 0 (the same numbering `build_mesh` printed, for example `idx=1201` for the outlet). The cell picks these automatically, and you can add more with `EXTRA_PIXEL_POINTS` and `EXTRA_QOUT_POINTS` in Settings:

| Label | Node chosen | Goes in |
|---|---|---|
| gauge | the **stream** node nearest the USGS gauge | both lists |
| mid_stream | the stream node about halfway between the gauge and the stream node farthest from it (straight-line) | both lists |
| centroid | the **interior** node nearest the basin centroid | pixel list |

(pytRIBS can also write node lists as X,Y coordinates instead of IDs; this notebook uses IDs, the form the SMF notebook used. After the first run, the `_voi` results file lists each ID with its x,y, which is a way to confirm the numbering.)

In [ ]:
verts = mesh_data['vertices']            # x, y, z for every node; the row number is the node ID
node_codes = mesh_data['node_codes']     # 0 interior, 1 boundary, 2 outlet, 3 stream


def nearest_node(x, y, allowed_codes):
    """Return (node ID, distance in m) of the node nearest (x, y) among nodes with the given codes."""
    ids = np.where(np.isin(node_codes, allowed_codes))[0]
    dist = np.hypot(verts[ids, 0] - x, verts[ids, 1] - y)
    k = int(np.argmin(dist))
    return int(ids[k]), float(dist[k])


# Gauge: nearest stream node
gauge_id, gauge_dist = nearest_node(OUTLET_X, OUTLET_Y, [3])

# Mid-stream: the stream node nearest the midpoint between the gauge node and the stream node farthest from it
stream_ids = np.where(node_codes == 3)[0]
d_from_gauge = np.hypot(verts[stream_ids, 0] - verts[gauge_id, 0], verts[stream_ids, 1] - verts[gauge_id, 1])
far_id = int(stream_ids[np.argmax(d_from_gauge)])
mid_x = (verts[gauge_id, 0] + verts[far_id, 0]) / 2
mid_y = (verts[gauge_id, 1] + verts[far_id, 1]) / 2
mid_id, mid_dist = nearest_node(mid_x, mid_y, [3])

# Centroid: nearest interior node to the basin centroid
centroid = tmesh.mesh_generator.watershed.geometry.iloc[0].centroid
centroid_id, centroid_dist = nearest_node(centroid.x, centroid.y, [0])

picks = [('gauge', gauge_id, gauge_dist, 'pixel + qout')]
pixel_nodes = [gauge_id]
qout_nodes = [gauge_id]

for k, (x, y) in enumerate(EXTRA_PIXEL_POINTS, start=1):
    node_id, dist = nearest_node(x, y, [0, 3])
    picks.append((f'extra_pixel_{k}', node_id, dist, 'pixel'))
    pixel_nodes.append(node_id)
for k, (x, y) in enumerate(EXTRA_QOUT_POINTS, start=1):
    node_id, dist = nearest_node(x, y, [3])
    picks.append((f'extra_qout_{k}', node_id, dist, 'qout'))
    qout_nodes.append(node_id)

pixel_nodes = list(dict.fromkeys(pixel_nodes))       # drop repeats, keep order
qout_nodes = list(dict.fromkeys(qout_nodes))

report = pd.DataFrame([{'label': label, 'node_id': nid, 'x': round(verts[nid, 0], 1), 'y': round(verts[nid, 1], 1),
                        'elev_m': round(verts[nid, 2], 1), 'code': code_names[int(node_codes[nid])],
                        'offset_from_target_m': round(dist, 1), 'output': out}
                       for label, nid, dist, out in picks])
display(report)

outlet_ids = np.where(node_codes == 2)[0]
print('For reference, the mesh outlet node (code 2) is ID', ', '.join(str(i) for i in outlet_ids),
      'at', ', '.join(f'({verts[i, 0]:.1f}, {verts[i, 1]:.1f})' for i in outlet_ids))

In [ ]:
# Write the node lists into the model folder and tell tRIBS where they are
pixel_file = os.path.join(proj.directories['model'], 'pnodes.nol')
qout_file = os.path.join(proj.directories['model'], 'qnodes.nol')
model.write_node_file([(OUTLET_X, OUTLET_Y)], pixel_file, coords=True)
model.write_node_file([(OUTLET_X, OUTLET_Y)], qout_file, coords=True)
model.nodeoutputlist['value'] = pixel_file
model.outletnodelist['value'] = qout_file
print('Pixel nodes          :', pixel_nodes, '->', pixel_file)
print('Internal flow nodes  :', qout_nodes, '->', qout_file)

# Where the picks are
fig, ax = plt.subplots(figsize=(8, 8))
tmesh.mesh_generator.watershed.boundary.plot(ax=ax, color='black', linewidth=1)
tmesh.mesh_generator.stream_network.plot(ax=ax, color='blue', linewidth=0.8)
ax.plot(verts[pixel_nodes, 0], verts[pixel_nodes, 1], 'o', color='orange', markeredgecolor='black', markersize=9, label='pixel nodes')
ax.plot(verts[qout_nodes, 0], verts[qout_nodes, 1], 's', color='red', markersize=5, label='internal flow nodes')
ax.plot([OUTLET_X], [OUTLET_Y], '*', color='yellow', markeredgecolor='black', markersize=16, label='gauge')
ax.set_title('Output nodes')
ax.legend(loc='upper left')
plt.show()

### Check every file path before writing

pytRIBS has a built-in `check_paths()`, but in pytRIBS 1.0.0 it only checks the station lists and the grid-list files, so the next cell runs it **and** a stricter check of its own. The stricter check covers every file the input file will point to, including the four mesh files, every grid in `scgrid.gdf`, every weather data file, and the radar files for the run window. It stops the notebook if anything is missing, so a bad input file is never written.

In [ ]:
model.check_paths()

In [ ]:
def check_inputs(model):
    """Check that every file the input file will point to exists. Returns a list of (status, item, detail)."""
    opt = model.options
    rows = []

    def add(ok, item, detail=''):
        rows.append(('OK' if ok else 'MISSING', item, detail))

    # Mesh: four files that share one base name
    base = opt['inputdatafile']['value']
    for ext in ('nodes', 'edges', 'tri', 'z'):
        add(os.path.isfile(f'{base}.{ext}'), f'mesh .{ext}', f'{base}.{ext}')

    # Single files
    for name in ('soiltablename', 'soilmapname', 'bedrockfile', 'gwaterfile', 'landtablename', 'landmapname',
                 'nodeoutputlist', 'outletnodelist'):
        path = opt[name]['value']
        add(path is not None and os.path.isfile(path), name, str(path))

    # Soil grid list and every grid it names
    scgrid = opt['scgrid']['value']
    add(scgrid is not None and os.path.isfile(scgrid), 'scgrid', str(scgrid))
    if scgrid is not None and os.path.isfile(scgrid):
        with open(scgrid) as fh:
            for row in csv.DictReader(fh):
                grid = f"{row['BasePath'].strip()}.{row['FileExtension'].strip()}"
                add(os.path.isfile(grid), f"soil grid {row['Variable'].strip()}", grid)

    # Weather station list and each station's data file
    sdf = opt['hydrometstations']['value']
    add(sdf is not None and os.path.isfile(sdf), 'hydrometstations', str(sdf))
    if sdf is not None and os.path.isfile(sdf):
        for s in model.read_met_sdf():
            add(os.path.isfile(s['file_path']), f"weather station {s['station_id']}", s['file_path'])

    # Radar rain: the files for this run window
    if int(opt['rainsource']['value']) == 1:
        rain_base = opt['rainfile']['value']
        folder, stem = os.path.split(rain_base)
        ext = opt['rainextension']['value']
        have = set(os.listdir(folder)) if os.path.isdir(folder) else set()
        start = datetime.strptime(opt['startdate']['value'], '%m/%d/%Y/%H/%M')
        wanted = [f"{stem}{(start + timedelta(hours=h)):%m%d%Y%H}.{ext}" for h in range(1, int(opt['runtime']['value']) + 1)]
        lacking = [w for w in wanted if w not in have]
        add(os.path.isdir(folder), 'radar folder', folder)
        add(not lacking, f'radar files for the run ({len(wanted)} needed)',
            'all present' if not lacking else f'{len(lacking)} missing, e.g. {lacking[:3]}')

    # Output folder
    out_folder = os.path.dirname(opt['outfilename']['value'])
    add(os.path.isdir(out_folder), 'results folder', out_folder)
    return rows


rows = check_inputs(model)
for status, item, detail in rows:
    print(f'{status:8s} {item:42s} {detail}')

n_missing = sum(1 for r in rows if r[0] == 'MISSING')
if n_missing:
    raise RuntimeError(f'{n_missing} input(s) are missing (listed above). Fix them before writing the input file.')
print(f'\nAll {len(rows)} checks passed.')

### Write the input file

This writes `Sycamore.in` in `lab/`. `Run_Model.ipynb` reads it.

In [ ]:
model.write_input_file(INPUT_FILE)
print('Wrote', os.path.abspath(INPUT_FILE))

### Read the input file back

The next cell reads the file that was just written and checks the settings that matter most for this build. Each `KEYWORD:` line in a tRIBS input file is followed by its value on the next line. A line marked CHECK is not necessarily wrong; it means the value differs from what this notebook expects, so look at it.

In [ ]:
def read_in_file(path):
    """Return {KEYWORD: value} from a tRIBS input file (each keyword line is followed by its value)."""
    with open(path) as fh:
        lines = fh.read().splitlines()
    found = {}
    for i, line in enumerate(lines):
        m = re.match(r'^([A-Z0-9]+):\s', line + ' ')
        if m and i + 1 < len(lines):
            found[m.group(1)] = lines[i + 1].strip()
    return found


def same(written, expected):
    """True if two values match as text, or as numbers when both are numbers."""
    try:
        return float(written) == float(expected)
    except (TypeError, ValueError):
        return str(written) == str(expected)


written = read_in_file(INPUT_FILE)
radar_prefix = f'{RADAR_DIR}/{RADAR_BASENAME}'

expected = [
    ('RAINSOURCE', 1, 'gridded (radar) rain'),
    ('RAINFILE', radar_prefix, 'radar folder + base name'),
    ('RAINEXTENSION', RADAR_EXT, 'radar file extension'),
    ('RAININTRVL', RAIN_INTERVAL_HR, 'hours between rain files'),
    ('METDATAOPTION', 1, 'weather station point data'),
    ('OPTSOILTYPE', 1, 'gridded soils'),
    ('OPTBEDROCK', 1, 'bedrock depth grid'),
    ('OPTGWFILE', 0, 'initial groundwater from a grid'),
    ('OPTLANDUSE', 0, 'land class map + table'),
    ('OPTMESHINPUT', 1, 'pre-built mesh'),
    ('OPTSNOW', OPTSNOW, 'snow module'),
    ('OPTPERCOLATION', OPTPERCOLATION, 'channel transmission loss'),
    ('CHANNELCONDUCTIVITY', CHANNELCONDUCTIVITY, 'mm/hr'),
    ('UTCOFFSET', UTC_OFFSET, 'hours from UTC'),
    ('STARTDATE', START_DATE, 'run start'),
    ('RUNTIME', RUN_HOURS, 'run length, hours'),
]
must_have_value = ['CENTROIDLAT', 'CENTROIDLONG', 'INPUTDATAFILE', 'SCGRID', 'SOILTABLENAME', 'SOILMAPNAME',
                   'BEDROCKFILE', 'GWATERFILE', 'LANDMAPNAME', 'LANDTABLENAME', 'HYDROMETSTATIONS', 'OUTFILENAME',
                   'NODEOUTPUTLIST', 'OUTLETNODELIST']

n_check = 0
for key, want, meaning in expected:
    got = written.get(key)
    ok = got is not None and same(got, want)
    n_check += (not ok)
    print(f"{'OK   ' if ok else 'CHECK'}  {key:20s} {str(got):45s} {meaning}")
for key in must_have_value:
    got = written.get(key)
    ok = bool(got)
    n_check += (not ok)
    print(f"{'OK   ' if ok else 'CHECK'}  {key:20s} {str(got):45s}")
print('\nGAUGESTATIONS (not used with gridded rain):', repr(written.get('GAUGESTATIONS')))
print('TEMPLAPSE (air temperature lapse rate, K/m):', written.get('TEMPLAPSE'))
print(f'\n{n_check} line(s) to look at.' if n_check else '\nEvery expected setting is in the input file.')

### Optional check: which bedrock and water-table files did the input file use?

Reads the `BEDROCKFILE` and `GWATERFILE` lines of the input file that was just written, prints when each file was last written and its minimum / median / maximum, and shows what the notebook holds in memory. It is useful when a run seems to be using an out-of-date grid (an early run did use a stale water-table file): if the "last written" time or the values look old, rerun the soil cells above.

In [ ]:
# Which bedrock and starting-water-table files does the input file point to, and what is in them?

def read_keywords(path):
    """Return {KEYWORD: value} from a tRIBS input file (each keyword line is followed by its value)."""
    with open(path) as fh:
        lines = fh.read().splitlines()
    found = {}
    for i, line in enumerate(lines):
        m = re.match(r'^([A-Z0-9]+):\s', line + ' ')
        if m and i + 1 < len(lines):
            found[m.group(1)] = lines[i + 1].strip()
    return found

written = read_keywords(INPUT_FILE)
print('Input file:', INPUT_FILE)
for key, unit in [('BEDROCKFILE', 'depth to bedrock, m'), ('GWATERFILE', 'starting water table depth, mm')]:
    path = written[key]
    grid = soil._read_ascii(path)
    nodata = grid['profile'].get('nodata', -9999)
    vals = grid['data'].astype(float)
    vals = vals[vals != nodata]
    print(f'\n{key}: {path}')
    print(f'  last written: {time.ctime(os.path.getmtime(path))}')
    print(f'  {unit}: min {vals.min():.2f}, median {np.median(vals):.2f}, max {vals.max():.2f}')

print('\nWhat the notebook currently holds in memory:')
print('  BEDROCK_ADD_M    =', BEDROCK_ADD_M)
print('  bedrock_path     =', bedrock_path)
print('  gw_path          =', gw_path)
print('  soil.bedrockfile =', soil.bedrockfile['value'])
print('  soil.gwaterfile  =', soil.gwaterfile['value'])

## Done: the input file is written

The file named in `INPUT_FILE` is now in `lab/`, and the read-back above checked its main settings.

**Run tRIBS from the terminal** (how the Sycamore calibration runs have been done), from `workspaces/Sycamore/lab`:
```
tRIBS <INPUT_FILE> 2>&1 | tee simulation_<run name>.log
```
A 4,368-hour run (Oct 1 to Apr 1) takes about 32 minutes. Keep the Codespace browser tab open while it runs, and do not run Settings or the build cells until it finishes. (`Run_Model.ipynb` is the other way to run tRIBS and read results.) When the run is done, open **`Compare_Sycamore_Runs.ipynb`** to compare it with the gauge.

**Worth checking whenever the inputs change:**
- tRIBS reading the mesh without complaint (about 15,800 nodes; the roughly 1,000 small triangles are the first suspect if it does complain).
- The startup printout should say **Gridded Rainfall**.
- Any message about the `9999.99` placeholder columns in `soils.sdt`, or about the `RZD_m` (root zone depth) column in the land table.
- Run time and the water balance in the `.mrf` file.
- Timing: compare the rain with the rise at the gauge for one clear storm (in the compare notebook). Rain should lead the rise by a plausible lag.